In [1]:
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader

load_dotenv()

file_path = "report.pdf"

loader = PyPDFLoader(file_path)
pages = []

async for page in loader .alazy_load():
    pages.append(page)

C:\Users\user\AppData\Local\Temp\ipykernel_25176\778322865.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
print("페이지 수: ", len(pages))
print("첫 번째 페이지 정보: ", pages[0])

페이지 수:  14
첫 번째 페이지 정보:  page_content='DATA STUDY  /  KOREAN EQUITIES  /  RAG
2026-10-06
한국 상장주식 RAG 실습 데이터
실제 상장종목 32개 | 8개 섹터 | KRX 종가 기준 | 작성·확인일 2026-10-07
교육/실습용 자료이며 투자 권유가 아님 | 기준일과 결측값 확인 필수
01 / 14
REAL STOCKS
SEARCHABLE FACTS
2026-10-06 종가 스냅샷
반도체 · 2차전지 · 자동차 · 바이오/헬스케어
금융 · 방산 · 인터넷/AI · 소비재
교육/실습용 자료이며 투자 권유가 아님
실제 시장자료를 정리한 학습 문서다. 종목의 매수·매도 추천이나 미래 수익 보장을 포함하지 않는다.
이 문서로 테스트할 수 있는 검색
• 반도체 섹터 중 종가 10만원 이하인 종목
• 방산 섹터 중 최근 3개월 수익률이 양수인 종목
• PBR 1배 이하인 금융주
• AI 관련주 중 비교 가능한 PER이 가장 낮은 종목
수록 범위: 종가·시가총액·밸류에이션·ROE·52주 범위·수익률·배당과 종목별 사업·테마·모멘텀·리스크. 표와 설명에 동일한 종목코
드와 출처 ID를 반복해 검색 근거를 연결한다.' metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-10-07T12:17:28+09:00', 'author': '데이터 공부 | 교육용 시장자료 정리', 'keywords': '', 'moddate': '2026-10-07T12:17:28+09:00', 'subject': 'unspecified', 'title': '한국 상장주식 RAG 실습 데이터 | 2026-10-06 종가', 'trapped': '/False', 'source': 'report.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 50)
docs = text_splitter.split_documents(pages)

In [4]:
print(f"총 {len(docs)}개의 청크 생성 완료")
print("각 청크의 길이: ", [len(i.page_content) for i in docs])

for i in docs:
    print("[메타데이터]", i.metadata)
    print("[내용]", i.page_content)
    print("=" * 100)

총 60개의 청크 생성 완료
각 청크의 길이:  [466, 149, 461, 424, 424, 494, 496, 498, 406, 499, 489, 495, 497, 119, 493, 454, 445, 492, 125, 497, 449, 494, 450, 75, 489, 492, 436, 492, 448, 467, 473, 493, 248, 475, 473, 489, 424, 496, 497, 461, 447, 75, 435, 486, 462, 453, 297, 481, 490, 427, 426, 75, 495, 488, 234, 496, 497, 487, 491, 97]
[메타데이터] {'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-10-07T12:17:28+09:00', 'author': '데이터 공부 | 교육용 시장자료 정리', 'keywords': '', 'moddate': '2026-10-07T12:17:28+09:00', 'subject': 'unspecified', 'title': '한국 상장주식 RAG 실습 데이터 | 2026-10-06 종가', 'trapped': '/False', 'source': 'report.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}
[내용] DATA STUDY  /  KOREAN EQUITIES  /  RAG
2026-10-06
한국 상장주식 RAG 실습 데이터
실제 상장종목 32개 | 8개 섹터 | KRX 종가 기준 | 작성·확인일 2026-10-07
교육/실습용 자료이며 투자 권유가 아님 | 기준일과 결측값 확인 필수
01 / 14
REAL STOCKS
SEARCHABLE FACTS
2026-10-06 종가 스냅샷
반도체 · 2차전지 · 자동차 · 바이오/헬스케어
금융 · 방산 · 인터넷/AI · 소비재
교육/실습용 자료이며 투자 권유가

In [5]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

DB_PATH = "./chroma_db"

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory=DB_PATH,
    collection_name="report_pdf",
)

In [6]:
vectorstore.similarity_search("PBR 1배 이하 금융주", k=3)

[Document(id='8876de2b-0e4b-47fa-b270-bf3921f07432', metadata={'title': '한국 상장주식 RAG 실습 데이터 | 2026-10-06 종가', 'trapped': '/False', 'author': '데이터 공부 | 교육용 시장자료 정리', 'creationdate': '2026-10-07T12:17:28+09:00', 'keywords': '', 'moddate': '2026-10-07T12:17:28+09:00', 'subject': 'unspecified', 'page_label': '1', 'producer': 'ReportLab PDF Library - (opensource)', 'source': 'report.pdf', 'total_pages': 14, 'creator': 'anonymous', 'page': 0}, page_content='• PBR 1배 이하인 금융주\n• AI 관련주 중 비교 가능한 PER이 가장 낮은 종목\n수록 범위: 종가·시가총액·밸류에이션·ROE·52주 범위·수익률·배당과 종목별 사업·테마·모멘텀·리스크. 표와 설명에 동일한 종목코\n드와 출처 ID를 반복해 검색 근거를 연결한다.'),
 Document(id='6f6f3a02-ea28-409f-8111-152c7624a97e', metadata={'keywords': '', 'total_pages': 14, 'producer': 'ReportLab PDF Library - (opensource)', 'creationdate': '2026-10-07T12:17:28+09:00', 'trapped': '/False', 'title': '한국 상장주식 RAG 실습 데이터 | 2026-10-06 종가', 'subject': 'unspecified', 'source': 'report.pdf', 'author': '데이터 공부 | 교육용 시장자료 정리', 'moddate': '2026-10-07T12:17:28+09:00', '